# E08 — Uncertainty and sensitivity
Independent Python formulation of the executed JavaScript stress analysis. Source mean ± derived SD bounds are not sampling confidence intervals. The envelope takes the union of source stress and model sensitivity; it is not a joint uncertainty propagation.


In [ ]:
from pathlib import Path
import subprocess
import numpy as np
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'outputs/E08/audit.json').exists():
    if (ROOT.parent/'outputs/E08/audit.json').exists(): ROOT=ROOT.parent
    else:
        ROOT=Path('/content/SeaweedMat')
        if not ROOT.exists(): subprocess.run(['git','clone','https://github.com/peterDataScientia/SeaweedMat.git',str(ROOT)],check=True)
read=lambda p: pd.read_csv(ROOT/p)
core=read('data/E02/mechanical_ageing_core_v0.0.1.csv').set_index('record_id')
m=read('outputs/E07/mechanical_pareto.csv')
k=read('outputs/E04/kinetic_sensitivity.csv')
b=read('outputs/E05/theobald_summary.csv').set_index('treatment')
c=read('outputs/E05/theobald_contrasts.csv')
e=read('outputs/E07/replacement_constraints.csv')
assert (len(core),len(m),len(k),len(b),len(c),len(e))==(71,9,46,10,16,60)
w=3/14
I=lambda a,b:(1-w)*a+w*b
rows=[]
for r in m.itertuples():
    pre='AQ19_'+r.configuration.replace(' ','_')+'_d'
    base,left,right=[core.loc[pre+str(t)] for t in [0,42,56]]
    flo=I(left.lower,right.lower)*9.80665
    fhi=I(left.upper,right.upper)*9.80665
    rlo=100*I(left.lower,right.lower)/base.upper
    rhi=100*I(left.upper,right.upper)/base.lower
    klo=min(r.retention_pct,r.model_min_pct)
    khi=max(r.retention_pct,r.model_max_pct)
    fklo=min(r.force_N,r.initial_force_N*klo/100)
    fkhi=max(r.force_N,r.initial_force_N*khi/100)
    rows.append(dict(configuration=r.configuration,nominal_retention_pct=r.retention_pct,nominal_force_N=r.force_N,source_stress_low_pct=rlo,source_stress_high_pct=rhi,source_stress_low_N=flo,source_stress_high_N=fhi,model_low_pct=klo,model_high_pct=khi,model_low_N=fklo,model_high_N=fkhi,envelope_low_pct=min(rlo,klo),envelope_high_pct=max(rhi,khi),envelope_low_N=min(flo,fklo),envelope_high_N=max(fhi,fkhi)))
M=pd.DataFrame(rows)
ref=read('outputs/E08/mechanical_stress_envelopes.csv')
j=M.merge(ref,on='configuration',validate='one_to_one',suffixes=('_rerun','_saved'))
for col in M.columns[1:]:
    np.testing.assert_allclose(j[col+'_rerun'],j[col+'_saved'],atol=1e-5,rtol=0)
display(M.round(2))


In [ ]:
tiers=['nominal','model_low','source_stress_low','envelope_low']
G=[]
for r in [60,70,80,90,100]:
    for f in [100,200,300,400,600]:
        for t in tiers:
            rc='nominal_retention_pct' if t=='nominal' else t+'_pct'
            fc='nominal_force_N' if t=='nominal' else t+'_N'
            a=M[(M[rc]>=r)&(M[fc]>=f)]
            G.append(dict(min_retention_pct=r,min_breaking_force_N=f,stress_case=t,n_candidates=len(a)))
G=pd.DataFrame(G)
ref=read('outputs/E08/constraint_stress_grid.csv')
j=G.merge(ref,on=['min_retention_pct','min_breaking_force_N','stress_case'],validate='one_to_one',suffixes=('_rerun','_saved'))
np.testing.assert_array_equal(j.n_candidates_rerun,j.n_candidates_saved)
print(G.groupby('stress_case').n_candidates.apply(lambda x:(x>0).sum()))
display(G[(G.min_retention_pct==80)&(G.min_breaking_force_N==200)])
possible=[]
for r in M.itertuples():
    other=M[M.configuration!=r.configuration]
    p=other[(other.envelope_high_pct>=r.envelope_low_pct)&(other.envelope_high_N>=r.envelope_low_N)&((other.envelope_high_pct>r.envelope_low_pct)|(other.envelope_high_N>r.envelope_low_N))]
    possible.append(len(p))
assert M.loc[np.array(possible)==0,'configuration'].tolist()==['Twisted jute 6']
# Model-specific fronts
for model in ['zero_order','first_order','weibull','lag_first_order']:
    a=k[(k.stratum=='exact_table_primary')&(k.model==model)].merge(m,left_on='label',right_on='configuration',validate='one_to_one')
    a['f']=a.predicted_retention_pct*a.initial_force_N/100
    def front(a):
        z=a[['predicted_retention_pct','f']].to_numpy()
        return set(a.loc[[not any(np.all(y>=x) and np.any(y>x) for y in z) for x in z],'configuration'])
    assert front(a)=={'Twisted jute 6'}
    assert front(a[a.material=='Cotton'])=={'Twisted cotton 3','Braided cotton 5'}


In [ ]:
c['direction']=np.select([c.boot_lo>0,c.boot_hi<0],['positive','negative'],default='crosses zero')
display(c[['material','alginate','endpoint','mean_difference','boot_lo','boot_hi','direction']])
for coat in ['',' + alginate']:
    x,y=b.loc['Cotton'+coat],b.loc['Polypropylene'+coat]
    d0=x.relative_cover_pct_mean-y.relative_cover_pct_mean
    d1=x.retention_sensitivity_mean-y.retention_sensitivity_mean
    assert d0*d1<0
    print(coat or 'uncoated',round(d0,3),round(d1,3),'remaining n:',x.retention_sensitivity_n,y.retention_sensitivity_n)
e['recalculated']=e.price_per_m_ratio/(e.lifetime_ratio*e.marketable_yield_ratio)
e['at_or_below']=e.recalculated<=1+1e-10
assert e.at_or_below.sum()==26
print(e.groupby('lifetime_ratio').at_or_below.agg(['sum','size']))
print('Numerical checks passed. Stress and scenario ranges are not probabilities.')


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family':'DejaVu Sans','font.weight':'bold','axes.labelweight':'bold','axes.labelsize':15,'font.size':12,'axes.linewidth':2,'xtick.major.width':2,'ytick.major.width':2,'xtick.major.size':6,'ytick.major.size':6,'axes.spines.top':False,'axes.spines.right':False,'legend.fontsize':11})
out=ROOT/'outputs/E08_python_rerun'
out.mkdir(parents=True,exist_ok=True)
fig,ax=plt.subplots(figsize=(170/25.4,145/25.4))
p=M.sort_values('nominal_retention_pct')
y=np.arange(len(p))
ax.hlines(y,p.envelope_low_pct,p.envelope_high_pct,color='#B15C00',lw=2.5,label='Stress envelope')
ax.scatter(p.nominal_retention_pct,y,color='#075DE8',s=45,zorder=3,label='Nominal')
ax.set_yticks(y,p.configuration)
ax.set_xlabel('Day-45 strength retention [%]')
ax.axvline(80,color='black',ls='--',lw=1.2)
ax.legend(frameon=False,loc='lower right')
fig.tight_layout()
for ext in ['png','svg','pdf']: fig.savefig(out/('mechanical_stress.'+ext),dpi=600,bbox_inches='tight',facecolor='white')
plt.show()
